In [1]:
import duckdb
import pandas as pd
from pathlib import Path
import diskcache
from itertools import chain

from utils.pandas_setup import pandas_setup
pandas_setup()

from unidecode import unidecode
from nameparser import HumanName

import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
pyalex.config.email = "Lawrence.Cram@anu.edu.au"
pyalex.config.max_retries = 0
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]

MY_DATA_PATH = Path('../DATA/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
MY_CACHE_FILE = Path('/home/lc/m/.cache/econommicsbusiness/cache.db')
DATAFILES_PATH = Path('../DATAFILES')

def normalise_name(in_name: str=None) -> str:
    in_name = ' '.join([part.strip() for part in unidecode(in_name).split(' ')])
    in_name = in_name.title()
    name = HumanName(in_name)
    if name.title == 'Md.Shahriar':
        name.first = 'Shahriar'
        name.title = ''
        # print(f'{name = }')
    if name.title == 'Mahdi':
        name.first = 'Mahdi'
        name.title = ''
        # print(f'{name = }')
    # print(f'{name = }')
    if name.middle:
        return f'{name.first} {name.middle} {name.last}'
    return f'{name.first} {name.last}'

In [2]:
class SetUp:

    def __init__(self):
        self._setup_db()
        self._setup_cache()
        return
    
    def _setup_db(self):
        self.db = duckdb.connect(MY_DATABASE_FILE)
        self.db.sql(""" SET memory_limit = '56GB';
                        SET threads = 6;
                        SET preserve_insertion_order = false;
                        SET order_by_non_integer_literal=true;
                        SET enable_progress_bar = true;
                        SET temp_directory = '/home/lc/m/.tmp';
                    """)
        self.db.sql("SHOW ALL TABLES").show()
        return
    
    def _setup_cache(self):
        self.cache = diskcache.Cache(MY_CACHE_FILE, size_limit=16_000_000_000)
        print(f'{self.cache.check() = }')
        print(f'{self.cache.volume() = }')
        return
    
    def name_of_global_obj(self, obj=None):
        for objname, oid in globals().items():
            if oid is obj:
                return objname

In [ ]:
class CorpusETL(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def extract_corpus(self, author_id=None):
        print(f'{author_id = }')
        print(Authors()[author_id])
        return
    
    def extract_corpus_endogenous(self, author_id=None):
        pick = f"WHERE author_id='{author_id}'" if author_id else ""
        sql = f"""
                SELECT *
                    FROM works w
                    RIGHT JOIN 
                        (SELECT * 
                            FROM authorships 
                            {pick}
                        )
                        USING (work_id)
                    WHERE contains('article review preprint letter', w.type) = true
            """
        df = self.db.sql(sql).df()
        print(df)
        return
    
    def extract_samples(self):
        self._extract_original_samples()
        self._match_samples()
        return
    
    def _extract_original_samples(self):
        sample = pd.read_excel('../RESULTS/researchers_results.xlsx').drop(columns=['Unnamed: 0', 'NAME'])
        print(f'{sample.shape = }\n{sample.head()}')
        sample['Research_Profile'] = [normalise_name(f"{n.split(', ')[1]} {n.split(', ')[0]}") if ',' in n else normalise_name(n) for n in sample.Research_Profile]
        sample = sample.sort_values("Research_Profile")
        print(f'{sample.shape = }\n{sample.head()}')
        self.db.sql("CREATE OR REPLACE TABLE econ.samples AS SELECT * FROM sample")
        self.db.sql("SELECT * FROM econ.samples").show()
        return
    
    def _match_samples(self):
        sql = """ 
            SELECT DISTINCT Research_Profile,
                    author_id,
                    author_name,
                    "Group", 
                    CIT,
                    count(work_id) AS works_count
                FROM econ.samples s
                    LEFT JOIN authorships a
                        ON a.author_name = s.Research_Profile
                    --WHERE a.author_name NOT NULL
                    GROUP BY ALL
                    ORDER BY author_name ASC, works_count DESC
            """
        df = self.db.sql(sql).df()
        df = df.drop_duplicates(subset='author_name')
        print(f'{df.shape = }\n{df.head()}')
        self.db.sql("CREATE OR REPLACE TABLE econ.sample_names AS SELECT * FROM df")
        return


In [4]:
def main():

    cetl = CorpusETL()
    # cetl.extract_corpus_endogenous() #(author_id='https://openalex.org/A5101600363')
    cetl.extract_samples()

    return

In [5]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────